# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import duckdb
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except:
    HF_TOKEN = os.environ.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print("✅ Connected to FlyRank Warehouse")

✅ Connected to FlyRank Warehouse


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
from IPython.display import Markdown, display

display(Markdown("""
### Data Contract

**Lane:** Refresh / Content Opportunity Scoring

**Unit of analysis**
- One row represents one content item (page) aggregated over a historical observation window for a single client.

**Primary table(s)**
- fact_content_daily_performance
- dim_content
- dim_clients

**Time window**
- Development and verification use a mid-panel month (March 2026) to avoid leakage from the final month.
- June 2026 is treated as a held-out period for final evaluation.

**Prediction / ranking target**
- Rank content items that should be refreshed based on historical search performance signals.

**Deliberately excluded**
- Any future-derived or label-derived fields (for example trend_direction and trend_pct), because they leak future information into the model.
"""))


### Data Contract

**Lane:** Refresh / Content Opportunity Scoring

**Unit of analysis**
- One row represents one content item (page) aggregated over a historical observation window for a single client.

**Primary table(s)**
- fact_content_daily_performance
- dim_content
- dim_clients

**Time window**
- Development and verification use a mid-panel month (March 2026) to avoid leakage from the final month.
- June 2026 is treated as a held-out period for final evaluation.

**Prediction / ranking target**
- Rank content items that should be refreshed based on historical search performance signals.

**Deliberately excluded**
- Any future-derived or label-derived fields (for example trend_direction and trend_pct), because they leak future information into the model.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
from IPython.display import Markdown, display

display(Markdown("""
## Field Classification

| Category | Fields | Why |
|----------|--------|-----|
| **Features** | gsc_impressions, gsc_clicks, gsc_avg_position, visible_query_count, rare_impressions_share | These are historical search signals available before making a refresh decision. |
| **Label / Proxy** | Refresh priority score (or future decline / opportunity ranking) | This is what the analysis attempts to rank or predict. It is never used as an input feature. |
| **Context** | client_hash_id, content_hash_id, report_date | Used for joining, grouping, filtering, and validation only. IDs are never model features. |
| **Excluded** | trend_direction, trend_pct, future-window metrics, private identifiers | These leak future information or should not be used as model inputs. |
"""))


## Field Classification

| Category | Fields | Why |
|----------|--------|-----|
| **Features** | gsc_impressions, gsc_clicks, gsc_avg_position, visible_query_count, rare_impressions_share | These are historical search signals available before making a refresh decision. |
| **Label / Proxy** | Refresh priority score (or future decline / opportunity ranking) | This is what the analysis attempts to rank or predict. It is never used as an input feature. |
| **Context** | client_hash_id, content_hash_id, report_date | Used for joining, grouping, filtering, and validation only. IDs are never model features. |
| **Excluded** | trend_direction, trend_pct, future-window metrics, private identifiers | These leak future information or should not be used as model inputs. |


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {TABLES['fact_daily']}
WHERE strftime(report_date, '%Y-%m') = '2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [9]:
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE strftime(report_date, '%Y-%m') = '2026-03'
  AND gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [11]:
from IPython.display import Markdown, display

display(Markdown("""
### Data Limitation

This dataset has an unbalanced history because different clients started providing GSC and GA4 data at different times. Some rows contain only GSC data, while GA4 metrics are unavailable until each client's GA4 start date.

Therefore, this analysis cannot assume every content item has the same historical coverage, and comparisons across clients should be interpreted carefully.
"""))


### Data Limitation

This dataset has an unbalanced history because different clients started providing GSC and GA4 data at different times. Some rows contain only GSC data, while GA4 metrics are unavailable until each client's GA4 start date.

Therefore, this analysis cannot assume every content item has the same historical coverage, and comparisons across clients should be interpreted carefully.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.